# License Plate Detection — Evaluation & Comparison
**CMPS 261 — Machine Learning Project**

Loads trained weights locally, runs inference on the test set, computes metrics, and generates comparison plots.

In [33]:
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'Running on: {"Google Colab" if IN_COLAB else "Local"}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile
    zip_path = '/content/drive/MyDrive/license_plate_data.zip'
    if not os.path.exists('/content/data/archive'):
        if not os.path.exists(zip_path):
            raise RuntimeError('license_plate_data.zip not found in Google Drive root.')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/')
        print('Extracted dataset archive.')
    BASE_DIR = '/content'
else:
    BASE_DIR = '..'

def prepare_purified_dataset(base_dir, seed=42):
    """Create a deduplicated YOLO split from data/archive."""
    import hashlib
    import random
    import shutil
    import xml.etree.ElementTree as ET
    from pathlib import Path

    base_dir = Path(base_dir)
    img_dir = base_dir / 'data' / 'archive' / 'images'
    ann_dir = base_dir / 'data' / 'archive' / 'annotations'
    yolo_dir = base_dir / 'data' / 'yolo'

    if not img_dir.exists() or not ann_dir.exists():
        raise RuntimeError(f'Raw dataset not found under {base_dir / "data" / "archive"}')

    def file_md5(path):
        h = hashlib.md5()
        with open(path, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 20), b''):
                h.update(chunk)
        return h.hexdigest()

    def parse_xml(xml_path):
        root = ET.parse(xml_path).getroot()
        filename = root.find('filename').text
        img_w = int(root.find('size/width').text)
        img_h = int(root.find('size/height').text)
        boxes = []
        for obj in root.findall('object'):
            boxes.append((
                int(obj.find('bndbox/xmin').text),
                int(obj.find('bndbox/ymin').text),
                int(obj.find('bndbox/xmax').text),
                int(obj.find('bndbox/ymax').text),
            ))
        return filename, img_w, img_h, boxes

    def voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h):
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        return cx, cy, w, h

    xml_files = sorted(ann_dir.glob('*.xml'))
    hash_to_xmls = {}
    for xml_path in xml_files:
        filename, *_ = parse_xml(xml_path)
        img_path = img_dir / filename
        if img_path.exists():
            hash_to_xmls.setdefault(file_md5(img_path), []).append(xml_path)

    unique_xmls = sorted(min(group) for group in hash_to_xmls.values())
    print(f'Dedup: {len(xml_files)} XMLs -> {len(unique_xmls)} unique images ({len(xml_files) - len(unique_xmls)} duplicate copies removed)')

    random.seed(seed)
    random.shuffle(unique_xmls)
    n = len(unique_xmls)
    n_train = int(n * 0.70)
    n_val = int(n * 0.15)
    splits = {
        'train': unique_xmls[:n_train],
        'val': unique_xmls[n_train:n_train + n_val],
        'test': unique_xmls[n_train + n_val:],
    }

    if yolo_dir.exists():
        shutil.rmtree(yolo_dir)
    for split in splits:
        (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    for split, files in splits.items():
        for xml_path in files:
            filename, img_w, img_h, boxes = parse_xml(xml_path)
            shutil.copy2(img_dir / filename, yolo_dir / 'images' / split / filename)
            label_path = yolo_dir / 'labels' / split / f'{Path(filename).stem}.txt'
            with open(label_path, 'w') as f:
                for xmin, ymin, xmax, ymax in boxes:
                    cx, cy, w, h = voc_to_yolo(xmin, ymin, xmax, ymax, img_w, img_h)
                    f.write(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n')

    yaml_path = yolo_dir / 'dataset.yaml'
    yaml_root = str(yolo_dir.resolve()) if str(base_dir) != '/content' else '/content/data/yolo'
    with open(yaml_path, 'w') as f:
        f.write(f'path: {yaml_root}\n')
        f.write('train: images/train\n')
        f.write('val:   images/val\n')
        f.write('test:  images/test\n\n')
        f.write('nc: 1\n')
        f.write("names: ['licence']\n")

    seen = {}
    for split in splits:
        for img in (yolo_dir / 'images' / split).iterdir():
            h = file_md5(img)
            if h in seen and seen[h] != split:
                raise RuntimeError(f'Cross-split duplicate after dedup: {img.name} in {split} matches {seen[h]}')
            seen[h] = split

    print('Data prepared:')
    for split, files in splits.items():
        print(f'  {split:<5}: {len(files)} images')
    print(f'  YAML  : {yaml_path}')
    print('  Cross-split duplicates: 0 (verified)')
    return str(yaml_path)

YAML_PATH = prepare_purified_dataset(BASE_DIR)

import os, certifi
os.environ.setdefault('SSL_CERT_FILE', certifi.where())
os.environ.setdefault('REQUESTS_CA_BUNDLE', certifi.where())
import json, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms.functional as TF

if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', 'pycocotools', '-q'], check=True)
    RESULTS_DIR = '/content/results'
    MODELS_DIR = '/content/models'
    VAL_IMG_DIR = '/content/data/yolo/images/val'
    VAL_LBL_DIR = '/content/data/yolo/labels/val'
    TEST_IMG_DIR = '/content/data/yolo/images/test'
    TEST_LBL_DIR = '/content/data/yolo/labels/test'
else:
    RESULTS_DIR = '../results'
    MODELS_DIR = '../models'
    VAL_IMG_DIR = '../data/yolo/images/val'
    VAL_LBL_DIR = '../data/yolo/labels/val'
    TEST_IMG_DIR = '../data/yolo/images/test'
    TEST_LBL_DIR = '../data/yolo/labels/test'

YOLO_IMGSZ = 960
DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

if IN_COLAB:
    import shutil
    for weight_name in ['yolov8s_best.pt', 'fasterrcnn_best.pth', 'retinanet_best.pth']:
        dst = os.path.join(MODELS_DIR, weight_name)
        candidates = [
            f'/content/drive/MyDrive/{weight_name}',
            f'/content/drive/MyDrive/models/{weight_name}',
        ]
        if not os.path.exists(dst):
            for src_path in candidates:
                if os.path.exists(src_path):
                    shutil.copy2(src_path, dst)
                    print(f'Copied {weight_name} to /content/models')
                    break

print(f'Device: {DEVICE}')


Device: mps


## 1. Evaluate YOLOv8s on Test Set

In [34]:
from ultralytics import YOLO
import yaml

yolo_model = YOLO(os.path.join(MODELS_DIR, 'yolov8s_best.pt'))

DATA_YAML = os.path.abspath(YAML_PATH)
DATA_DIR  = os.path.abspath(os.path.dirname(YAML_PATH))
EVAL_DATA_YAML = os.path.abspath(os.path.join(RESULTS_DIR, 'dataset_eval.yaml'))

# Write an evaluation-only YAML so the tracked dataset.yaml is not mutated.
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = DATA_DIR
with open(EVAL_DATA_YAML, 'w') as f:
    yaml.dump(cfg, f)

yolo_val = yolo_model.val(
    data=EVAL_DATA_YAML,
    split='test',
    imgsz=YOLO_IMGSZ,
    verbose=False
)

yolo_metrics = {
    'model'      : 'YOLOv8s',
    'imgsz'      : YOLO_IMGSZ,
    'precision'  : round(float(yolo_val.box.mp),    4),
    'recall'     : round(float(yolo_val.box.mr),    4),
    'map50'      : round(float(yolo_val.box.map50), 4),
    'map50_95'   : round(float(yolo_val.box.map),   4),
    'f1'         : round(2 * float(yolo_val.box.mp) * float(yolo_val.box.mr) /
                        (float(yolo_val.box.mp) + float(yolo_val.box.mr) + 1e-6), 4),
}

print(json.dumps(yolo_metrics, indent=2))

# Save
with open(os.path.join(RESULTS_DIR, 'yolo_metrics.json'), 'w') as f:
    json.dump(yolo_metrics, f, indent=2)

Ultralytics 8.4.37 🚀 Python-3.13.5 torch-2.11.0 CPU (Apple M4)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1114.3±317.4 MB/s, size: 515.2 KB)
val: Scanning /Users/ward/Desktop/aub/261/project/261_code/license plate/data/yolo/labels/test.cache... 66 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 66/66 16.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 5.1s/it 25.3s8.5sss
                   all         66         71       0.97       0.91      0.942      0.517
Speed: 1.0ms preprocess, 372.9ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /Users/ward/Desktop/aub/261/project/261_code/license plate/notebooks/runs/detect/val5
{
  "model": "YOLOv8s",
  "imgsz": 960,
  "precision": 0.97,
  "recall": 0.9103,
  "map50": 0.9418,
  "map50_95": 0.5168,
  "f1": 0.9392
}


## 2. Evaluate Faster R-CNN on Test Set

In [35]:
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir):
        self.samples = []
        for lbl_file in sorted(os.listdir(lbl_dir)):
            if not lbl_file.endswith('.txt'):
                continue
            stem = os.path.splitext(lbl_file)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = os.path.join(img_dir, stem + ext)
                if os.path.exists(img_path):
                    self.samples.append((img_path, os.path.join(lbl_dir, lbl_file)))
                    break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, lbl_path = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        boxes = []
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                _, cx, cy, w, h = map(float, parts[:5])
                xmin = max(0.0, (cx - w/2) * W)
                ymin = max(0.0, (cy - h/2) * H)
                xmax = min(float(W), (cx + w/2) * W)
                ymax = min(float(H), (cy + h/2) * H)
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
        if not boxes:
            boxes = [[0.0, 0.0, 1.0, 1.0]]
        boxes  = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.ones(len(boxes), dtype=torch.int64)
        return TF.to_tensor(img), {'boxes': boxes, 'labels': labels}

def collate_fn(batch):
    return tuple(zip(*batch))

# Load model
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
frcnn_model = fasterrcnn_resnet50_fpn_v2(weights=weights)
in_features = frcnn_model.roi_heads.box_predictor.cls_score.in_features
frcnn_model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
frcnn_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'fasterrcnn_best.pth'), map_location=DEVICE, weights_only=True))
frcnn_model.to(DEVICE).eval()

val_ds      = LicensePlateDataset(VAL_IMG_DIR,  VAL_LBL_DIR)
test_ds     = LicensePlateDataset(TEST_IMG_DIR, TEST_LBL_DIR)
val_loader  = DataLoader(val_ds,  batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
print(f'Val: {len(val_ds)} | Test: {len(test_ds)} images')

def compute_iou(a, b):
    xA, yA = max(a[0],b[0]), max(a[1],b[1])
    xB, yB = min(a[2],b[2]), min(a[3],b[3])
    inter  = max(0, xB-xA) * max(0, yB-yA)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-6)

def collect_predictions(model, loader):
    cache = []
    model.eval()
    with torch.no_grad():
        for images, targets in loader:
            images = [img.to(DEVICE) for img in images]
            preds = model(images)
            for pred, target in zip(preds, targets):
                scores = pred['scores'].detach().cpu().numpy()
                order = np.argsort(-scores)
                cache.append({
                    'boxes': pred['boxes'].detach().cpu().numpy()[order],
                    'scores': scores[order],
                    'gt_boxes': target['boxes'].numpy(),
                })
    return cache

def eval_cached(cache, threshold):
    tp, fp, fn = 0, 0, 0
    iou_scores = []
    for item in cache:
        gt_boxes = item['gt_boxes']
        pred_boxes = item['boxes'][item['scores'] >= threshold]
        matched = set()
        for pb in pred_boxes:
            best_iou, best_j = 0, -1
            for j, gb in enumerate(gt_boxes):
                if j in matched:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou: best_iou, best_j = iou, j
            if best_iou >= 0.5 and best_j != -1:
                tp += 1; matched.add(best_j); iou_scores.append(best_iou)
            else:
                fp += 1
        fn += len(gt_boxes) - len(matched)
    p  = tp / (tp + fp + 1e-6)
    r  = tp / (tp + fn + 1e-6)
    f1 = 2 * p * r / (p + r + 1e-6)
    return p, r, f1, float(np.mean(iou_scores)) if iou_scores else 0.0

def find_best_threshold(cache, min_threshold=0.05, max_threshold=0.99):
    score_arrays = [item['scores'] for item in cache if len(item['scores'])]
    if not score_arrays:
        return min_threshold, eval_cached(cache, min_threshold)
    scores = np.concatenate(score_arrays)
    candidates = np.unique(scores[(scores >= min_threshold) & (scores <= max_threshold)])
    candidates = np.unique(np.concatenate(([min_threshold, max_threshold], candidates)))
    best_thresh, best_metrics = min_threshold, eval_cached(cache, min_threshold)
    for thresh in candidates:
        metrics = eval_cached(cache, float(thresh))
        if (metrics[2], metrics[0], float(thresh)) > (best_metrics[2], best_metrics[0], best_thresh):
            best_thresh, best_metrics = float(thresh), metrics
    return best_thresh, best_metrics

def eval_thresh(model, loader, threshold):
    return eval_cached(collect_predictions(model, loader), threshold)

print('Finding exact best Faster R-CNN threshold on validation scores...')
frcnn_val_cache = collect_predictions(frcnn_model, val_loader)
best_thresh, frcnn_val_metrics = find_best_threshold(frcnn_val_cache)
print(f'Best validation F1 threshold: {best_thresh:.4f}')
print(f'Val P={frcnn_val_metrics[0]:.4f} R={frcnn_val_metrics[1]:.4f} F1={frcnn_val_metrics[2]:.4f}')

frcnn_test_cache = collect_predictions(frcnn_model, test_loader)
precision, recall, f1, mean_iou = eval_cached(frcnn_test_cache, best_thresh)
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Mean IoU  : {mean_iou:.4f}')

frcnn_metrics = {
    'model'    : 'Faster R-CNN (ResNet50-FPN v2)',
    'threshold': round(float(best_thresh), 4),
    'precision': round(precision, 4),
    'recall'   : round(recall,    4),
    'f1'       : round(f1,        4),
    'mean_iou' : round(mean_iou,  4),
}
print(json.dumps(frcnn_metrics, indent=2))

with open(os.path.join(RESULTS_DIR, 'fasterrcnn_metrics.json'), 'w') as f:
    json.dump(frcnn_metrics, f, indent=2)

Val: 64 | Test: 66 images
Finding exact best Faster R-CNN threshold on validation scores...
Best validation F1 threshold: 0.8281
Val P=0.9286 R=0.9286 F1=0.9286
Precision : 0.9143
Recall    : 0.9014
F1        : 0.9078
Mean IoU  : 0.7905
{
  "model": "Faster R-CNN (ResNet50-FPN v2)",
  "threshold": 0.8281,
  "precision": 0.9143,
  "recall": 0.9014,
  "f1": 0.9078,
  "mean_iou": 0.7905
}


In [36]:
import ssl, certifi, urllib.request, os

url = 'https://download.pytorch.org/models/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth'
out = os.path.expanduser('~/.cache/torch/hub/checkpoints/retinanet_resnet50_fpn_v2_coco-5905b1c5.pth')
os.makedirs(os.path.dirname(out), exist_ok=True)

if not os.path.exists(out):
    ssl_ctx = ssl.create_default_context(cafile=certifi.where())
    urllib.request.urlretrieve(url, out)
    print('Downloaded.')
else:
    print('Already cached.')

Already cached.


## 3. Evaluate RetinaNet on Test Set

In [37]:
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

weights = RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT
retina_model = retinanet_resnet50_fpn_v2(weights=weights)
num_anchors  = retina_model.head.classification_head.num_anchors
in_channels  = retina_model.head.classification_head.conv[0][0].in_channels
retina_model.head.classification_head = RetinaNetClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=2,
    norm_layer=torch.nn.BatchNorm2d,
)
retina_model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'retinanet_best.pth'), map_location=DEVICE, weights_only=True))
retina_model.to(DEVICE).eval()
print(f'Val: {len(val_ds)} | Test: {len(test_ds)} images')

print('Finding exact best RetinaNet threshold on validation scores...')
retina_val_cache = collect_predictions(retina_model, val_loader)
best_thresh, retina_val_metrics = find_best_threshold(retina_val_cache)
print(f'Best validation F1 threshold: {best_thresh:.4f}')
print(f'Val P={retina_val_metrics[0]:.4f} R={retina_val_metrics[1]:.4f} F1={retina_val_metrics[2]:.4f}')

retina_test_cache = collect_predictions(retina_model, test_loader)
precision, recall, f1, mean_iou = eval_cached(retina_test_cache, best_thresh)
print(f'Precision : {precision:.4f}')
print(f'Recall    : {recall:.4f}')
print(f'F1        : {f1:.4f}')
print(f'Mean IoU  : {mean_iou:.4f}')

retina_metrics = {
    'model'    : 'RetinaNet (ResNet50-FPN v2)',
    'threshold': round(float(best_thresh), 4),
    'precision': round(precision, 4),
    'recall'   : round(recall,    4),
    'f1'       : round(f1,        4),
    'mean_iou' : round(mean_iou,  4),
}
print(json.dumps(retina_metrics, indent=2))

with open(os.path.join(RESULTS_DIR, 'retinanet_metrics.json'), 'w') as f:
    json.dump(retina_metrics, f, indent=2)

Val: 64 | Test: 66 images
Finding exact best RetinaNet threshold on validation scores...
Best validation F1 threshold: 0.4745
Val P=0.9385 R=0.8714 F1=0.9037
Precision : 0.8784
Recall    : 0.9155
F1        : 0.8966
Mean IoU  : 0.7823
{
  "model": "RetinaNet (ResNet50-FPN v2)",
  "threshold": 0.4745,
  "precision": 0.8784,
  "recall": 0.9155,
  "f1": 0.8966,
  "mean_iou": 0.7823
}


In [38]:
yolo_df = pd.DataFrame([
    {
        'Model'       : 'YOLOv8s',
        'Precision'   : yolo_metrics['precision'],
        'Recall'      : yolo_metrics['recall'],
        'F1'          : yolo_metrics['f1'],
        'mAP@0.5'     : yolo_metrics['map50'],
        'mAP@0.5:0.95': yolo_metrics['map50_95'],
        'Evaluator'   : 'Ultralytics val()',
    }
]).set_index('Model')

torchvision_df = pd.DataFrame([
    {
        'Model'       : 'RetinaNet',
        'Threshold'   : retina_metrics['threshold'],
        'Precision'   : retina_metrics['precision'],
        'Recall'      : retina_metrics['recall'],
        'F1'          : retina_metrics['f1'],
        'Mean IoU'    : retina_metrics['mean_iou'],
        'Evaluator'   : 'Custom greedy IoU>=0.5',
    },
    {
        'Model'       : 'Faster R-CNN',
        'Threshold'   : frcnn_metrics['threshold'],
        'Precision'   : frcnn_metrics['precision'],
        'Recall'      : frcnn_metrics['recall'],
        'F1'          : frcnn_metrics['f1'],
        'Mean IoU'    : frcnn_metrics['mean_iou'],
        'Evaluator'   : 'Custom greedy IoU>=0.5',
    },
]).set_index('Model').sort_values('F1', ascending=False)

print('YOLO metrics are reported from Ultralytics. Torchvision metrics use a custom IoU>=0.5 evaluator with thresholds selected on validation F1.')
print('\nYOLOv8s')
print(yolo_df.to_string())
print('\nTorchvision models')
print(torchvision_df.to_string())
yolo_df, torchvision_df

YOLO metrics are reported from Ultralytics. Torchvision metrics use a custom IoU>=0.5 evaluator with thresholds selected on validation F1.

YOLOv8s
         Precision  Recall      F1  mAP@0.5  mAP@0.5:0.95          Evaluator
Model                                                                       
YOLOv8s       0.97  0.9103  0.9392   0.9418        0.5168  Ultralytics val()

Torchvision models
              Threshold  Precision  Recall      F1  Mean IoU               Evaluator
Model                                                                               
Faster R-CNN     0.8281     0.9143  0.9014  0.9078    0.7905  Custom greedy IoU>=0.5
RetinaNet        0.4745     0.8784  0.9155  0.8966    0.7823  Custom greedy IoU>=0.5


(         Precision  Recall      F1  mAP@0.5  mAP@0.5:0.95          Evaluator
 Model                                                                       
 YOLOv8s       0.97  0.9103  0.9392   0.9418        0.5168  Ultralytics val(),
               Threshold  Precision  Recall      F1  Mean IoU  \
 Model                                                          
 Faster R-CNN     0.8281     0.9143  0.9014  0.9078    0.7905   
 RetinaNet        0.4745     0.8784  0.9155  0.8966    0.7823   
 
                            Evaluator  
 Model                                 
 Faster R-CNN  Custom greedy IoU>=0.5  
 RetinaNet     Custom greedy IoU>=0.5  )

In [39]:
import os
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# YOLO is plotted separately because these are Ultralytics detector metrics.
yolo_labels = ['Precision', 'Recall', 'F1', 'mAP@0.5', 'mAP@0.5:0.95']
yolo_vals = [
    yolo_metrics['precision'], yolo_metrics['recall'], yolo_metrics['f1'],
    yolo_metrics['map50'], yolo_metrics['map50_95'],
]
bars = axes[0].bar(yolo_labels, yolo_vals, color='#4C9BE8', alpha=0.85)
for bar, val in zip(bars, yolo_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=8)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('Score')
axes[0].set_title('YOLOv8s — Ultralytics Metrics')
axes[0].tick_params(axis='x', rotation=25)

labels = ['Precision', 'Recall', 'F1', 'Mean IoU']
retina_vals = [retina_metrics['precision'], retina_metrics['recall'], retina_metrics['f1'], retina_metrics['mean_iou']]
frcnn_vals = [frcnn_metrics['precision'], frcnn_metrics['recall'], frcnn_metrics['f1'], frcnn_metrics['mean_iou']]
x, width = np.arange(len(labels)), 0.35
bars1 = axes[1].bar(x - width/2, retina_vals, width, label=f"RetinaNet @ {retina_metrics['threshold']:.2f}", color='#6BCB77', alpha=0.85)
bars2 = axes[1].bar(x + width/2, frcnn_vals, width, label=f"Faster R-CNN @ {frcnn_metrics['threshold']:.2f}", color='#E87B4C', alpha=0.85)
for bar in list(bars1) + list(bars2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Torchvision Models — Custom IoU>=0.5 F1')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Evaluation Summary — Metrics Shown by Evaluator Family', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'), dpi=150)
plt.show()
print('Saved: results/model_comparison.png')

<Figure size 1300x500 with 2 Axes>

Saved: results/model_comparison.png


test_images = random.sample(os.listdir(TEST_IMG_DIR), 4)
fig, axes   = plt.subplots(4, 4, figsize=(18, 16))
yolo_conf = 0.25
retina_conf = retina_metrics['threshold']
frcnn_conf = frcnn_metrics['threshold']

for row_idx, fname in enumerate(test_images):
    img_path = os.path.join(TEST_IMG_DIR, fname)
    img_pil  = Image.open(img_path).convert('RGB')
    img_np   = np.array(img_pil)

    axes[row_idx][0].imshow(img_np)
    axes[row_idx][0].set_title('Original', fontsize=9)
    axes[row_idx][0].axis('off')

    # YOLOv8s
    result = yolo_model.predict(img_path, conf=yolo_conf, verbose=False)[0]
    axes[row_idx][1].imshow(img_np)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        axes[row_idx][1].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none'))
        axes[row_idx][1].text(x1, y1-4, f'{box.conf[0]:.2f}', color='lime', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][1].set_title('YOLOv8s', fontsize=9)
    axes[row_idx][1].axis('off')

    # RetinaNet
    tensor = TF.to_tensor(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = retina_model(tensor)[0]
    axes[row_idx][2].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < retina_conf: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        axes[row_idx][2].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='#6BCB77', facecolor='none'))
        axes[row_idx][2].text(x1, y1-4, f'{score:.2f}', color='#6BCB77', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][2].set_title('RetinaNet', fontsize=9)
    axes[row_idx][2].axis('off')

    # Faster R-CNN
    with torch.no_grad():
        pred = frcnn_model(tensor)[0]
    axes[row_idx][3].imshow(img_np)
    for box, score in zip(pred['boxes'], pred['scores']):
        if score < frcnn_conf: continue
        x1, y1, x2, y2 = box.cpu().tolist()
        axes[row_idx][3].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='#FF6B6B', facecolor='none'))
        axes[row_idx][3].text(x1, y1-4, f'{score:.2f}', color='#FF6B6B', fontsize=8,
                              bbox=dict(facecolor='black', alpha=0.4, pad=1))
    axes[row_idx][3].set_title('Faster R-CNN', fontsize=9)
    axes[row_idx][3].axis('off')

plt.suptitle('Side-by-Side: Original | YOLOv8s | RetinaNet | Faster R-CNN', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'side_by_side_comparison.png'), dpi=150)
plt.show()
print('Saved: results/side_by_side_comparison.png')

## 5. Unified COCO-Style Evaluation

This runs `src/unified_evaluation.py`, which evaluates YOLOv8s, Faster R-CNN, and RetinaNet through the same COCO AP pipeline. It also tunes F1 thresholds on validation for all three models and bootstraps test F1 confidence intervals.

In [40]:
import os
import subprocess
import sys

script_path = '../src/unified_evaluation.py'
if not os.path.exists(script_path):
    print('Skipping unified_evaluation.py because the src folder is not available in this Colab notebook upload.')
else:
    cmd = [
        sys.executable,
        script_path,
        '--imgsz', str(YOLO_IMGSZ),
        '--bootstrap', '1000',
        '--output', '../results/unified_metrics.json',
    ]
    subprocess.run(cmd, check=True)


Device: mps
Validation images: 64 | Test images: 66


YOLO predictions: 100%|██████████| 66/66 [00:06<00:00, 10.74it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.511
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.938
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.519
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.412
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.571
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.416
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.613
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.615
 Average Recall     (AR) @[ IoU=0.5

Faster R-CNN predictions: 100%|██████████| 66/66 [00:26<00:00,  2.48it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.500
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.960
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.445
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.481
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.567
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.346
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.552
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.601
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.601
 Average Recall     (AR) @[ IoU=0.5

RetinaNet predictions: 100%|██████████| 66/66 [00:13<00:00,  4.81it/s]


creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.01s).
Accumulating evaluation results...
DONE (t=0.00s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.507
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.905
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.555
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.544
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.541
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.606
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.606
 Average Recall     (AR) @[ IoU=0.5

CompletedProcess(args=['/usr/local/bin/python3', '../src/unified_evaluation.py', '--imgsz', '960', '--bootstrap', '1000', '--output', '../results/unified_metrics.json'], returncode=0)

## 6. Split Near-Duplicate Check

This checks whether visually similar images appear across train/validation/test splits. Warnings here do not prove leakage, but they identify examples to inspect manually.

In [41]:
import os
import subprocess
import sys

script_path = '../src/check_split_duplicates.py'
if not os.path.exists(script_path):
    print('Skipping check_split_duplicates.py because the src folder is not available in this Colab notebook upload.')
else:
    cmd = [
        sys.executable,
        script_path,
        '--max-distance', '5',
        '--output', '../results/split_duplicate_report.json',
    ]
    subprocess.run(cmd, check=True)


Images checked: 433
Cross-split near-duplicate warnings: 102
Saved: ../results/split_duplicate_report.json


CompletedProcess(args=['/usr/local/bin/python3', '../src/check_split_duplicates.py', '--max-distance', '5', '--output', '../results/split_duplicate_report.json'], returncode=0)